In [ ]:
# NOTEBOOK PARAMETERS
selected_model = 'neptune_1'
ensemble_size = 8
years = [2019,2020,2021]

In [ ]:
import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd
import xarray as xr
import numpy as np
import os
import sys

sys.path.append('library')
from atmocean.visualize import plot_isotherm_20C, discrete_cmap
from atmocean.enso import compute_isotherm_20C
from atmocean.utils import clean_forecasts, remove_ocean_variables_suffix
os.system('hostname')
%load_ext autoreload
%autoreload 2

In [ ]:
# get the forecasting horizon for this run
start_date = '01-01'
end_date = '12-31'
lead_times = [0, 10, 20, 30, 40, 50, 60]

main_dir = 'PATH-TO-MAIN-DIRECTORY'

data_dir = os.path.join(main_dir, 'data')
forecast_dir = os.path.join(data_dir, 'forecasts')

if ensemble_size is None:
    forecasts_srcs = [
        os.path.join(forecast_dir, f'{selected_model}-from_{year}-{start_date}_to_{year}-{end_date}.zarr')
        for year in years
    ]
    out_img_dir = os.path.join(main_dir, f'validation/img/{selected_model}/s2s-{years[0]}-{years[-1]}/ocean_indices/iso20')
else:
    forecasts_srcs = [
        os.path.join(forecast_dir, f'{selected_model}-from_{year}-{start_date}_to_{year}-{end_date}-ENS-{ensemble_size:02d}.zarr')
        for year in years
    ]
    out_img_dir = os.path.join(main_dir, f'validation/img/{selected_model}/s2s-{years[0]}-{years[-1]}-ENS-{ensemble_size:02d}/ocean_indices/iso20')

os.makedirs(out_img_dir, exist_ok=True)
plot_kwargs = dict(dpi=100, bbox_inches='tight')
save_figures = True

In [ ]:
from dask.distributed import Client, LocalCluster, progress, wait
from tqdm import tqdm
import numpy as np
import dask
import ctypes

# spill to disk before hitting hard memory limits (avoids KilledWorker)
dask.config.set({
    'distributed.worker.memory.target': 0.6,
    'distributed.worker.memory.spill': 0.7,
    'distributed.worker.memory.pause': 0.8,
    'distributed.worker.memory.terminate': 0.95,
})

n_workers = 4
memory_limit = 948 // n_workers
cluster = LocalCluster(
    n_workers=n_workers, threads_per_worker=1,
    memory_limit=f'{memory_limit}GB',
    local_directory=os.path.join(main_dir, 'tmp_dask'),  # spill directory
)
client = Client(cluster)

def _trim_memory() -> int:
    libc = ctypes.CDLL("libc.so.6")
    return libc.malloc_trim(0)

client.run(_trim_memory)
client

In [ ]:
# open forecasts
forecasts = xr.open_mfdataset(
    forecasts_srcs, chunks={}, parallel=True, engine='zarr', preprocess=clean_forecasts,
    combine='nested', concat_dim='time', data_vars='minimal', coords='minimal', compat='override',
)
# forecasts = xr.open_mfdataset(forecasts_srcs, chunks=chunks, parallel=True, engine='zarr', preprocess=clean_forecasts)
# manipulate the time dimension to align
if 'highres' not in selected_model:
    forecasts = forecasts.isel(prediction_timedelta=lead_times)
forecasts = forecasts.isel(level=0, missing_dims='ignore')
# remove ocean variables suffix (oras, cglo, glor)
forecasts = remove_ocean_variables_suffix(forecasts)

In [ ]:
box_type = 'tight' # @parameter ['tight', 'wide']

west_box_wide = dict(latitude=slice(5, -5), longitude=slice(160, 180))
west_box_tight = dict(latitude=slice(2, -2), longitude=slice(165, 175))

east_box_wide = dict(latitude=slice(5, -5), longitude=slice(250, 270))
east_box_tight = dict(latitude=slice(2, -2), longitude=slice(260, 270))

if box_type == 'tight':
    east_box = east_box_tight
    west_box = west_box_tight
elif box_type == 'wide':
    east_box = east_box_wide
    west_box = west_box_wide

In [ ]:
forecasts = forecasts['thetao']

forecasts_east = forecasts.sel(east_box)
forecasts_west = forecasts.sel(west_box)

# compute now the average cause we have smaller slices
if ensemble_size is not None:
    forecasts_east = forecasts_east.mean(dim='ensemble_member', skipna=True)
    forecasts_west = forecasts_west.mean(dim='ensemble_member', skipna=True)

In [ ]:
futures = client.persist([
    forecasts_east,
    forecasts_west,
])
progress(futures)
wait(futures)

In [ ]:
forecasts_east = forecasts_east.load()
forecasts_west = forecasts_west.load()

In [ ]:
Z20_fcs_east = compute_isotherm_20C(forecasts_east, 200.0)
Z20_fcs_west = compute_isotherm_20C(forecasts_west, 200.0)

In [ ]:
colors = discrete_cmap(len(lead_times)+4, 'gnuplot2') # 'Blues_r')
colors = [colors(i) for i in range(len(lead_times)+1)]

In [ ]:
model_names = [
    'Reanalysis' if lead_time == 0 else f"Forecast t + {lead_time}"
    for lead_time in lead_times
]
model_colors = [
    colors[i+1] if i != 0 else 'lime'
    for i in range(len(lead_times))
]
zorders = [10] + [1] * len(lead_times)

In [ ]:
Z20s_east = []
# Z20s_east.append(Z20_clm_east.isel(prediction_timedelta=0))
for i,lt in enumerate(lead_times):
    Z20 = Z20_fcs_east.isel(prediction_timedelta=i)
    Z20['time'] = pd.to_datetime(Z20['time'].data) + pd.DateOffset(days=lt)
    Z20s_east.append(Z20)

outfile = os.path.join(out_img_dir, 'isotherm_20C_east.png') if save_figures else None
plot_isotherm_20C(
    Z20s=Z20s_east,
    model_names=model_names,
    model_colors=model_colors,
    zorders=zorders,
    box=dict(type='East', coords='5°S – 5°N, 100°W – 90°W'),
    sscale=0.5,
    use_anom=False,
    outfile=outfile,
)

In [ ]:
Z20s_west = []
# Z20s_west.append(Z20_clm_west.isel(prediction_timedelta=0))
for i,lt in enumerate(lead_times):
    Z20 = Z20_fcs_west.isel(prediction_timedelta=i)
    Z20['time'] = pd.to_datetime(Z20['time'].data) + pd.DateOffset(days=lt)
    Z20s_west.append(Z20)

outfile = os.path.join(out_img_dir, 'isotherm_20C_west.png')
plot_isotherm_20C(
    Z20s=Z20s_west,
    model_names=model_names,
    model_colors=model_colors,
    zorders=zorders,
    box=dict(type='West', coords='5°S – 5°N, 165°E – 175°E'),
    sscale=0.5,
    use_anom=False,
    outfile=outfile,
)